# Beyond Image Classification

Build a tiny semantic-segmentation model on generated rectangle images. Each output pixel receives a foreground/background score.

In [ ]:
import torch
from torch import nn

torch.manual_seed(36)
torch.set_num_threads(1)
print('CPU | synthetic pixel segmentation')

In [ ]:
def rectangles(count):
    images = torch.zeros(count, 1, 16, 16)
    masks = torch.zeros_like(images)
    for index in range(count):
        top = 2 + index % 6
        left = 2 + (index * 3) % 6
        height = 4 + index % 4
        width = 4 + (index + 1) % 4
        masks[index, 0, top:top+height, left:left+width] = 1
        images[index] = masks[index] + 0.05 * torch.randn(1, 16, 16)
    return images, masks

images, masks = rectangles(64)
print('images/masks:', tuple(images.shape), tuple(masks.shape))

In [ ]:
model = nn.Sequential(nn.Conv2d(1, 8, 3, padding=1), nn.ReLU(),
                      nn.Conv2d(8, 8, 3, padding=1), nn.ReLU(), nn.Conv2d(8, 1, 1))
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)
logits = model(images)
loss = nn.functional.binary_cross_entropy_with_logits(logits, masks)
optimizer.zero_grad()
loss.backward()
optimizer.step()
with torch.no_grad():
    predicted = model(images) > 0
    intersection = (predicted & masks.bool()).sum().item()
    union = (predicted | masks.bool()).sum().item()
    iou = intersection / union
assert logits.shape == masks.shape and torch.isfinite(loss)
print(f'pixel loss={loss.item():.4f} | training IoU after one update={iou:.3f}')